In [0]:
# 01 - Importar bibliotecas e carregar a base

from pyspark.sql import functions as F

from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer,
    OneHotEncoder,
    VectorAssembler
)
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

df = spark.table(
    "workspace.projeto_integrador.gold_features_ml"
)

print("Total de registros:", df.count())

In [0]:
# 02 - Criar conjunto de desenvolvimento e teste final

df_desenvolvimento = (
    df
    .filter(F.col("ano") <= 2024)
)

df_teste_final = (
    df
    .filter(F.col("ano") == 2025)
)

print(
    "Registros para treinamento final:",
    df_desenvolvimento.count()
)

print(
    "Registros para teste final:",
    df_teste_final.count()
)

In [0]:
# 03 - Criar função de avaliação

def calcular_metricas(df_predicoes, prediction_col):
    
    mae = RegressionEvaluator(
        labelCol="target",
        predictionCol=prediction_col,
        metricName="mae"
    ).evaluate(df_predicoes)

    rmse = RegressionEvaluator(
        labelCol="target",
        predictionCol=prediction_col,
        metricName="rmse"
    ).evaluate(df_predicoes)

    r2 = RegressionEvaluator(
        labelCol="target",
        predictionCol=prediction_col,
        metricName="r2"
    ).evaluate(df_predicoes)

    return mae, rmse, r2

In [0]:
# 04 - Avaliar o baseline no teste final de 2025

predicoes_baseline_2025 = (
    df_teste_final
    .withColumn(
        "prediction_baseline",
        F.col("valor_ano_anterior")
    )
)

mae_baseline_2025, rmse_baseline_2025, r2_baseline_2025 = (
    calcular_metricas(
        predicoes_baseline_2025,
        "prediction_baseline"
    )
)

print("BASELINE - TESTE 2025")
print("MAE:", round(mae_baseline_2025, 2))
print("RMSE:", round(rmse_baseline_2025, 2))
print("R²:", round(r2_baseline_2025, 4))

In [0]:
# 05 - Preparar pipeline da Regressão Linear final

indexador_uf = StringIndexer(
    inputCol="uf",
    outputCol="uf_index",
    handleInvalid="keep"
)

indexador_regiao = StringIndexer(
    inputCol="regiao_brasil",
    outputCol="regiao_index",
    handleInvalid="keep"
)

encoder = OneHotEncoder(
    inputCols=[
        "uf_index",
        "regiao_index"
    ],
    outputCols=[
        "uf_ohe",
        "regiao_ohe"
    ]
)

features_linear = [
    "valor_ano_anterior",
    "valor_2_anos_antes",
    "valor_3_anos_antes",
    "media_3_anos_anteriores",
    "diferenca_ultimo_ano",
    "variacao_percentual_historica",
    "razao_ultimo_media_3_anos",
    "uf_ohe",
    "regiao_ohe"
]

assembler = VectorAssembler(
    inputCols=features_linear,
    outputCol="features"
)

regressao_linear = LinearRegression(
    featuresCol="features",
    labelCol="target",
    predictionCol="prediction_ml",
    maxIter=100,
    regParam=0.1,
    elasticNetParam=0.0
)

pipeline_final = Pipeline(
    stages=[
        indexador_uf,
        indexador_regiao,
        encoder,
        assembler,
        regressao_linear
    ]
)

In [0]:
# 06 - Treinar modelo final e realizar previsões para 2025

modelo_final = pipeline_final.fit(
    df_desenvolvimento
)

predicoes_ml_2025 = (
    modelo_final
    .transform(df_teste_final)

    .withColumn(
        "prediction_ml",
        F.greatest(
            F.col("prediction_ml"),
            F.lit(0.0)
        )
    )
)

mae_ml_2025, rmse_ml_2025, r2_ml_2025 = calcular_metricas(
    predicoes_ml_2025,
    "prediction_ml"
)

print("REGRESSÃO LINEAR - TESTE 2025")
print("MAE:", round(mae_ml_2025, 2))
print("RMSE:", round(rmse_ml_2025, 2))
print("R²:", round(r2_ml_2025, 4))

In [0]:
# 07 - Comparar modelo de ML e baseline no teste final

resultados_finais = [
    (
        "Baseline - Ano anterior",
        float(mae_baseline_2025),
        float(rmse_baseline_2025),
        float(r2_baseline_2025)
    ),
    (
        "Regressão Linear - ML",
        float(mae_ml_2025),
        float(rmse_ml_2025),
        float(r2_ml_2025)
    )
]

df_resultados_finais = spark.createDataFrame(
    resultados_finais,
    [
        "modelo",
        "mae",
        "rmse",
        "r2"
    ]
)

df_resultados_finais = (
    df_resultados_finais
    .withColumn(
        "mae",
        F.round("mae", 2)
    )
    .withColumn(
        "rmse",
        F.round("rmse", 2)
    )
    .withColumn(
        "r2",
        F.round("r2", 4)
    )
    .orderBy("mae")
)

display(df_resultados_finais)

In [0]:
# 08 - Criar tabela detalhada das previsões de 2025

previsoes_finais = (
    predicoes_ml_2025
    .select(
        "codigo_ibge",
        "municipio",
        "uf",
        "estado",
        "regiao_brasil",
        "ano",
        "target",
        "valor_ano_anterior",
        "prediction_ml"
    )

    .withColumn(
        "prediction_baseline",
        F.col("valor_ano_anterior")
    )

    .withColumn(
        "erro_absoluto_ml",
        F.abs(
            F.col("target") -
            F.col("prediction_ml")
        )
    )

    .withColumn(
        "erro_absoluto_baseline",
        F.abs(
            F.col("target") -
            F.col("prediction_baseline")
        )
    )

    .withColumn(
        "erro_percentual_ml",
        F.when(
            F.col("target") > 0,
            F.round(
                (
                    F.col("erro_absoluto_ml") /
                    F.col("target")
                ) * 100,
                2
            )
        )
    )

    .withColumn(
        "erro_percentual_baseline",
        F.when(
            F.col("target") > 0,
            F.round(
                (
                    F.col("erro_absoluto_baseline") /
                    F.col("target")
                ) * 100,
                2
            )
        )
    )
)

display(
    previsoes_finais
    .orderBy(F.desc("target"))
    .limit(30)
)

In [0]:
# 09 - Salvar previsões finais no Unity Catalog

previsoes_finais.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "workspace.projeto_integrador.gold_previsoes_ml"
    )

print("Tabela gold_previsoes_ml salva com sucesso.")

In [0]:
# 10 - Validar tabela de previsões

gold_previsoes = spark.table(
    "workspace.projeto_integrador.gold_previsoes_ml"
)

print(
    "Quantidade de previsões:",
    gold_previsoes.count()
)

print(
    "Quantidade de municípios:",
    gold_previsoes
    .select("codigo_ibge")
    .distinct()
    .count()
)

display(
    gold_previsoes
    .orderBy(F.desc("target"))
    .limit(20)
)